In [1]:
import duckdb

S_LST = "../data/silver/nyc/listings.parquet"
S_CAL = "../data/silver/nyc/calendar.parquet"
S_REV = "../data/silver/nyc/reviews.parquet"

duckdb.sql(f"""
    SELECT
        (SELECT COUNT(*) FROM read_parquet('{S_LST}'))                                   AS listings,
        (SELECT COUNT(price) FROM read_parquet('{S_LST}'))                               AS listings_with_price,
        (SELECT COUNT(*) FROM read_parquet('{S_LST}') WHERE maximum_nights = 2147483647) AS placeholders_left,
        (SELECT COUNT(*) FROM read_parquet('{S_LST}') WHERE rating_overall = 0)          AS zero_ratings_left,
        (SELECT MIN(quote_discount) FROM read_parquet('{S_LST}'))                        AS min_discount,
        (SELECT COUNT(*) FROM read_parquet('{S_LST}')
         WHERE pre_discount_nightly_price < price - 0.01)                                AS pre_below_price,
        (SELECT COUNT(DISTINCT listing_id) FROM read_parquet('{S_CAL}'))                 AS calendar_listings,
        (SELECT MAX(days_from_start) FROM read_parquet('{S_CAL}'))                       AS max_days_from_start,
        (SELECT COUNT(*) FROM read_parquet('{S_REV}'))                                   AS reviews
""").show()

┌──────────┬─────────────────────┬───────────────────┬───────────────────┬──────────────┬─────────────────┬───────────────────┬─────────────────────┬─────────┐
│ listings │ listings_with_price │ placeholders_left │ zero_ratings_left │ min_discount │ pre_below_price │ calendar_listings │ max_days_from_start │ reviews │
│  int64   │        int64        │       int64       │       int64       │    double    │      int64      │       int64       │        int64        │  int64  │
├──────────┼─────────────────────┼───────────────────┼───────────────────┼──────────────┼─────────────────┼───────────────────┼─────────────────────┼─────────┤
│    30259 │               21515 │                 0 │                 0 │          0.0 │               2 │             30259 │                 365 │  980552 │
└──────────┴─────────────────────┴───────────────────┴───────────────────┴──────────────┴─────────────────┴───────────────────┴─────────────────────┴─────────┘



In [2]:
duckdb.sql(f"""
    SELECT
        listing_id,
        price,
        quote_total,
        quote_nights,
        quote_discount,
        pre_discount_nightly_price,
        quote_total / quote_nights                     AS total_per_night_unrounded,
        ROUND(price - pre_discount_nightly_price, 4)   AS gap
    FROM read_parquet('{S_LST}')
    WHERE pre_discount_nightly_price < price - 0.01
""").df()

,listing_id,price,quote_total,quote_nights,quote_discount,pre_discount_nightly_price,total_per_night_unrounded,gap
0,29906282,139.80,279.59,2,0.0,139.79,139.795,0.01
1,746530698054367292,318.16,9544.65,30,0.0,318.15,318.155,0.01
